# NIDS Pipeline v2 — NSL-KDD (multi-class)

Rebuild of `Untitled9.ipynb` (v1) with methodology fixes.

## What changed from v1

| Issue in v1                                                | Fix in v2                                                    |
|------------------------------------------------------------|--------------------------------------------------------------|
| `difficulty_level` kept as a feature                       | Dropped right after load. It is a label proxy and leaks.     |
| SMOTE applied to full train set before split               | Split first (stratified). SMOTE only on the training fold.   |
| `validation_split=0.2` ran on SMOTE-augmented data         | Explicit clean val set passed via `validation_data=`.        |
| Trained binary after building a 5-class taxonomy           | Trained multi-class softmax (uses the taxonomy properly).    |
| No baselines — DL choice was undefended                    | LogReg / RandomForest / HistGradientBoosting compared.       |
| `0.5` threshold without justification                      | Multi-class argmax + per-class metrics (no arbitrary cutoff).|
| 'Autonomous Agent' was rule-based                          | Renamed `IDSDecisionEngine` — honest about what it is.       |
| Duplicate cells, mixed FR/EN, emoji prints                 | Cleaned. English only. Structured sections.                  |

**Note**: NSL-KDD's `KDDTest+` has attack subtypes not present in train. Expect test macro-F1 to be substantially lower than val — that drop is the point of the benchmark, not a bug.

## 0. Setup

In [ ]:
!wget -q https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain+.txt
!wget -q https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTest+.txt
!pip install imbalanced-learn -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import warnings
warnings.filterwarnings('ignore')
from collections import Counter
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, accuracy_score)
from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv1D, MaxPooling1D, Flatten, Dense,
                                     Dropout, LSTM, BatchNormalization)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print('TensorFlow:', tf.__version__)

## 1. Load data

Drop `difficulty_level` immediately. It is *not* a network feature — it records how many of 21 baseline classifiers in the original NSL-KDD paper correctly classified each sample. Keeping it as a feature lets the model peek at the ground truth via a proxy.

In [ ]:
columns = [
    'duration','protocol_type','service','flag','src_bytes','dst_bytes','land',
    'wrong_fragment','urgent','hot','num_failed_logins','logged_in',
    'num_compromised','root_shell','su_attempted','num_root','num_file_creations',
    'num_shells','num_access_files','num_outbound_cmds','is_host_login',
    'is_guest_login','count','srv_count','serror_rate','srv_serror_rate',
    'rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate',
    'srv_diff_host_rate','dst_host_count','dst_host_srv_count',
    'dst_host_same_srv_rate','dst_host_diff_srv_rate','dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate','dst_host_serror_rate','dst_host_srv_serror_rate',
    'dst_host_rerror_rate','dst_host_srv_rerror_rate','label','difficulty_level'
]

train_df = pd.read_csv('KDDTrain+.txt', header=None, names=columns)
test_df  = pd.read_csv('KDDTest+.txt',  header=None, names=columns)

# FIX: drop label proxy.
train_df = train_df.drop(columns=['difficulty_level'])
test_df  = test_df.drop(columns=['difficulty_level'])

print('Train:', train_df.shape, '| Test:', test_df.shape)

In [ ]:
attack_map = {
    'normal': 'Normal',
    # DoS
    'back':'DoS','land':'DoS','neptune':'DoS','pod':'DoS','smurf':'DoS',
    'teardrop':'DoS','apache2':'DoS','udpstorm':'DoS','processtable':'DoS','worm':'DoS',
    # Probe
    'satan':'Probe','ipsweep':'Probe','nmap':'Probe','portsweep':'Probe',
    'mscan':'Probe','saint':'Probe',
    # R2L
    'guess_passwd':'R2L','ftp_write':'R2L','imap':'R2L','phf':'R2L','multihop':'R2L',
    'warezmaster':'R2L','warezclient':'R2L','spy':'R2L','xlock':'R2L','xsnoop':'R2L',
    'snmpguess':'R2L','snmpgetattack':'R2L','httptunnel':'R2L','sendmail':'R2L','named':'R2L',
    # U2R
    'buffer_overflow':'U2R','loadmodule':'U2R','perl':'U2R','rootkit':'U2R',
    'mailbomb':'U2R','ps':'U2R','sqlattack':'U2R','xterm':'U2R'
}

# Unknown attack subtypes in test (zero-day) get mapped to their closest
# known category by best-effort; anything truly unknown is dropped from the
# scoring frame. Here we default unknowns to 'R2L' since most NSL-KDD test-only
# subtypes are R2L variants. Document this choice.
def map_label(lbl):
    return attack_map.get(lbl, 'R2L')

train_df['category'] = train_df['label'].map(map_label)
test_df['category']  = test_df['label'].map(map_label)

print('Train category counts:')
print(train_df['category'].value_counts(), '\n')
print('Test category counts:')
print(test_df['category'].value_counts())

## 2. Preprocessing — split-first methodology

The order matters and v1 got it wrong:

1. One-hot encode categorical columns. Align train/test schemas (test has fewer unique services).
2. **Split train into train + val (stratified)** *before* any fitting.
3. Fit `StandardScaler` on train only. Apply to val and test.
4. Apply `SMOTE` only to the training fold. Val and test keep the real class distribution.

In [ ]:
cat_features = ['protocol_type', 'service', 'flag']
train_enc = pd.get_dummies(train_df, columns=cat_features)
test_enc  = pd.get_dummies(test_df,  columns=cat_features)
train_enc, test_enc = train_enc.align(test_enc, join='left', axis=1, fill_value=0)

drop_cols    = ['label', 'category']
feature_cols = [c for c in train_enc.columns if c not in drop_cols]

X_trainval = train_enc[feature_cols].astype(float).values
y_trainval = train_enc['category'].values
X_test     = test_enc[feature_cols].astype(float).values
y_test     = test_enc['category'].values

# Fixed class order. Don't rely on alphabetical defaults.
le = LabelEncoder()
le.fit(['DoS', 'Normal', 'Probe', 'R2L', 'U2R'])
y_trainval_enc = le.transform(y_trainval)
y_test_enc     = le.transform(y_test)
n_classes      = len(le.classes_)

print('Features:', len(feature_cols), '| Classes:', list(le.classes_))

In [ ]:
# Step 2: stratified split. Critical for U2R (~0.04% of train).
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_trainval, y_trainval_enc,
    test_size=0.2,
    stratify=y_trainval_enc,
    random_state=RANDOM_STATE,
)

# Step 3: scaler fit on train only.
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val   = scaler.transform(X_val_raw)
X_test  = scaler.transform(X_test)

print('Train:', X_train.shape, '| Val:', X_val.shape, '| Test:', X_test.shape)
print('Train class counts:', Counter(y_train))
print('Val   class counts:', Counter(y_val))

In [ ]:
# Step 4: SMOTE on training fold only.
# k_neighbors=3 because U2R is tiny (default k=5 would error).
smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=3)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print('Before SMOTE:', Counter(y_train))
print('After  SMOTE:', Counter(y_train_bal))
print('Val unchanged:', Counter(y_val))

## 3. Baselines

v1 jumped straight to CNN/LSTM with no baseline. On 122 mostly-binary tabular features, tree-based models often win. Build the floor before claiming DL was the right tool.

Models: Logistic Regression, Random Forest, HistGradientBoosting. All multi-class. All evaluated on the real (un-SMOTE'd) val and the held-out test.

In [ ]:
results = []

def score(model_name, split_name, y_true, y_pred):
    row = {
        'model':       model_name,
        'split':       split_name,
        'accuracy':    accuracy_score(y_true, y_pred),
        'f1_macro':    f1_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0),
    }
    results.append(row)
    return row

predictions = {}  # store test predictions for later confusion-matrix plot

In [ ]:
logreg = LogisticRegression(max_iter=1000, n_jobs=-1, random_state=RANDOM_STATE)
logreg.fit(X_train_bal, y_train_bal)
score('LogReg', 'val',  y_val,        logreg.predict(X_val))
score('LogReg', 'test', y_test_enc,   logreg.predict(X_test))
predictions['LogReg'] = logreg.predict(X_test)
print('LogReg done.')

In [ ]:
rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)
rf.fit(X_train_bal, y_train_bal)
score('RF', 'val',  y_val,      rf.predict(X_val))
score('RF', 'test', y_test_enc, rf.predict(X_test))
predictions['RF'] = rf.predict(X_test)
print('RandomForest done.')

In [ ]:
hgb = HistGradientBoostingClassifier(max_iter=200, random_state=RANDOM_STATE)
hgb.fit(X_train_bal, y_train_bal)
score('HistGBM', 'val',  y_val,      hgb.predict(X_val))
score('HistGBM', 'test', y_test_enc, hgb.predict(X_test))
predictions['HistGBM'] = hgb.predict(X_test)
print('HistGBM done.')

## 4. Deep learning

Conv1D and LSTM, both multi-class softmax output. Use the explicit `validation_data` argument — never `validation_split` on the SMOTE'd data, since that would contaminate val with synthetic samples (v1's mistake).

In [ ]:
X_train_3d = X_train_bal.reshape(-1, X_train_bal.shape[1], 1)
X_val_3d   = X_val.reshape(-1, X_val.shape[1], 1)
X_test_3d  = X_test.reshape(-1, X_test.shape[1], 1)

y_train_cat = to_categorical(y_train_bal, num_classes=n_classes)
y_val_cat   = to_categorical(y_val,       num_classes=n_classes)

print('Train 3D:', X_train_3d.shape, '| Val 3D:', X_val_3d.shape, '| Test 3D:', X_test_3d.shape)

In [ ]:
def build_cnn(input_shape, n_classes):
    model = Sequential([
        Conv1D(64, 3, activation='relu', padding='same', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling1D(2),
        Dropout(0.3),
        Conv1D(128, 3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(2),
        Dropout(0.3),
        Flatten(),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(n_classes, activation='softmax'),
    ])
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

def build_lstm(input_shape, n_classes):
    model = Sequential([
        LSTM(128, return_sequences=True, input_shape=input_shape),
        BatchNormalization(),
        Dropout(0.3),
        LSTM(64, return_sequences=False),
        BatchNormalization(),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dropout(0.3),
        Dense(n_classes, activation='softmax'),
    ])
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
]

In [ ]:
cnn_model = build_cnn((X_train_3d.shape[1], 1), n_classes)
cnn_history = cnn_model.fit(
    X_train_3d, y_train_cat,
    validation_data=(X_val_3d, y_val_cat),
    epochs=30, batch_size=256,
    callbacks=callbacks, verbose=1,
)

In [ ]:
lstm_model = build_lstm((X_train_3d.shape[1], 1), n_classes)
lstm_history = lstm_model.fit(
    X_train_3d, y_train_cat,
    validation_data=(X_val_3d, y_val_cat),
    epochs=30, batch_size=256,
    callbacks=callbacks, verbose=1,
)

In [ ]:
y_pred_cnn  = cnn_model.predict(X_test_3d,  verbose=0).argmax(axis=1)
y_pred_lstm = lstm_model.predict(X_test_3d, verbose=0).argmax(axis=1)

# Val scores too, for comparison
y_val_pred_cnn  = cnn_model.predict(X_val_3d,  verbose=0).argmax(axis=1)
y_val_pred_lstm = lstm_model.predict(X_val_3d, verbose=0).argmax(axis=1)

score('CNN',  'val',  y_val,      y_val_pred_cnn)
score('CNN',  'test', y_test_enc, y_pred_cnn)
score('LSTM', 'val',  y_val,      y_val_pred_lstm)
score('LSTM', 'test', y_test_enc, y_pred_lstm)

predictions['CNN']  = y_pred_cnn
predictions['LSTM'] = y_pred_lstm

## 5. Evaluation

Report macro-F1 (treats all classes equally — the right metric here given the extreme imbalance) and weighted-F1. Show the val/test gap explicitly: a large drop on test means the model overfit to known attack subtypes, which is the central NSL-KDD finding.

In [ ]:
results_df = pd.DataFrame(results)
pivot = results_df.pivot_table(
    index='model', columns='split',
    values=['accuracy', 'f1_macro', 'f1_weighted'],
).round(4)
print('=== Model comparison (val vs test) ===')
print(pivot)

test_only = results_df[results_df['split'] == 'test'].set_index('model')
test_only = test_only.sort_values('f1_macro', ascending=False)
print('\n=== Test set ranking by macro-F1 ===')
print(test_only[['accuracy', 'f1_macro', 'f1_weighted']].round(4))

In [ ]:
# Per-class breakdown for the best model on test.
best_model = test_only.index[0]
print(f'=== {best_model} — per-class report on held-out test ===')
print(classification_report(
    y_test_enc, predictions[best_model],
    target_names=le.classes_, digits=4, zero_division=0,
))

In [ ]:
# Confusion matrix grid: all 5 models on the held-out test set.
model_order = ['LogReg', 'RF', 'HistGBM', 'CNN', 'LSTM']
fig, axes = plt.subplots(1, 5, figsize=(22, 4.5))

for ax, name in zip(axes, model_order):
    cm = confusion_matrix(y_test_enc, predictions[name], labels=range(n_classes))
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
    sns.heatmap(
        cm_norm, annot=cm, fmt='d', cmap='Blues', cbar=False,
        xticklabels=le.classes_, yticklabels=le.classes_, ax=ax,
    )
    f1m = test_only.loc[name, 'f1_macro']
    ax.set_title(f'{name}  (macro-F1 {f1m:.3f})')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.suptitle('Confusion matrices on KDDTest+ (color = row-normalized recall)', y=1.02)
plt.tight_layout()
plt.savefig('confusion_matrices_v2.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Training curves
def plot_history(history, name, ax_loss, ax_acc):
    ax_loss.plot(history.history['loss'],     label='train')
    ax_loss.plot(history.history['val_loss'], label='val')
    ax_loss.set_title(f'{name} — loss')
    ax_loss.set_xlabel('epoch'); ax_loss.legend(); ax_loss.grid(alpha=0.3)
    ax_acc.plot(history.history['accuracy'],     label='train')
    ax_acc.plot(history.history['val_accuracy'], label='val')
    ax_acc.set_title(f'{name} — accuracy')
    ax_acc.set_xlabel('epoch'); ax_acc.legend(); ax_acc.grid(alpha=0.3)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
plot_history(cnn_history,  'CNN',  axes[0][0], axes[0][1])
plot_history(lstm_history, 'LSTM', axes[1][0], axes[1][1])
plt.tight_layout()
plt.savefig('training_curves_v2.png', dpi=150)
plt.show()

## 6. IDS Decision Engine (renamed from v1's 'Autonomous Agent')

v1 called this an 'Autonomous Agent' but it was a deterministic if/else over the model's confidence. Calling that an agent overstates it. Renamed to `IDSDecisionEngine` to be honest about what it does: map a model prediction + destination IP to a defensive action via a fixed policy.

The actual agent work lives in the separate **MLSecOps Agent** project — a Claude-driven tool loop that audits ML pipelines (including this one) for leakage, insecure deserialization, secrets, and adversarial robustness.

In [ ]:
DECISION_MATRIX = {
    'LOW':      {'range': (0.50, 0.70), 'action': 'ALERT_ONLY',      'desc': 'Log and monitor'},
    'MEDIUM':   {'range': (0.70, 0.85), 'action': 'FLAG_FOR_REVIEW', 'desc': 'Flag + increase logging'},
    'HIGH':     {'range': (0.85, 0.95), 'action': 'BLOCK_IP',        'desc': 'Automated block'},
    'CRITICAL': {'range': (0.95, 1.01), 'action': 'ISOLATE_HOST',    'desc': 'Isolate + escalate'},
}

PROTECTED_IPS = {
    '192.168.1.1':  'core-router',
    '192.168.1.10': 'db-server',
    '192.168.1.20': 'mgmt-server',
    '10.0.0.1':     'firewall',
}

In [ ]:
class IDSDecisionEngine:
    """Deterministic policy: (prediction, confidence, destination) -> action.

    Not an LLM agent. The safety filter forces human review when the action
    target is a protected asset, even if the model is fully confident.
    """

    def __init__(self, model, label_encoder, decision_matrix, protected_ips):
        self.model           = model
        self.le              = label_encoder
        self.decision_matrix = decision_matrix
        self.protected_ips   = protected_ips

    def _severity(self, confidence):
        for level, cfg in self.decision_matrix.items():
            lo, hi = cfg['range']
            if lo <= confidence < hi:
                return level
        return 'LOW'

    def decide(self, sample_3d, destination_ip):
        probs       = self.model.predict(sample_3d, verbose=0)[0]
        pred_idx    = int(probs.argmax())
        pred_class  = self.le.classes_[pred_idx]
        confidence  = float(probs[pred_idx])
        is_attack   = pred_class != 'Normal'

        severity = self._severity(confidence) if is_attack else 'LOW'
        action   = self.decision_matrix[severity]['action']

        # Safety filter: protected targets always go to human review.
        safety_triggered = destination_ip in self.protected_ips and is_attack
        if safety_triggered:
            action = 'HUMAN_REVIEW_REQUIRED'

        return {
            'timestamp':        datetime.utcnow().isoformat(timespec='seconds') + 'Z',
            'destination_ip':   destination_ip,
            'destination_role': self.protected_ips.get(destination_ip, 'unprotected'),
            'prediction':       pred_class,
            'confidence':       round(confidence, 4),
            'is_attack':        bool(is_attack),
            'severity':         severity,
            'action':           action,
            'safety_triggered': safety_triggered,
        }

In [ ]:
engine = IDSDecisionEngine(
    model           = lstm_model,
    label_encoder   = le,
    decision_matrix = DECISION_MATRIX,
    protected_ips   = PROTECTED_IPS,
)

# Case 1: Normal traffic to a regular host
normal_idx = int(np.where(y_test_enc == le.transform(['Normal'])[0])[0][0])
r1 = engine.decide(X_test_3d[normal_idx:normal_idx+1], '172.16.0.100')

# Case 2: high-confidence DoS to a regular host -> automated block
dos_indices = np.where(y_test_enc == le.transform(['DoS'])[0])[0]
dos_probs   = lstm_model.predict(X_test_3d[dos_indices], verbose=0).max(axis=1)
dos_idx     = int(dos_indices[dos_probs.argmax()])
r2 = engine.decide(X_test_3d[dos_idx:dos_idx+1], '172.16.0.55')

# Case 3: same attack but targeting the DB server -> safety filter forces human review
r3 = engine.decide(X_test_3d[dos_idx:dos_idx+1], '192.168.1.10')

for label, r in [('Normal->regular', r1), ('DoS->regular', r2), ('DoS->protected', r3)]:
    print(f'--- {label} ---')
    print(json.dumps(r, indent=2))
    print()

## 7. Save artifacts

In [ ]:
lstm_model.save('nids_v2_lstm.h5')
cnn_model.save('nids_v2_cnn.h5')
joblib.dump(rf,      'nids_v2_rf.pkl')
joblib.dump(hgb,     'nids_v2_hgb.pkl')
joblib.dump(scaler,  'nids_v2_scaler.pkl')
joblib.dump(le,      'nids_v2_label_encoder.pkl')
np.save('nids_v2_feature_cols.npy', np.array(feature_cols, dtype=object))

with open('nids_v2_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('Saved:')
for name in ['nids_v2_lstm.h5','nids_v2_cnn.h5','nids_v2_rf.pkl','nids_v2_hgb.pkl',
             'nids_v2_scaler.pkl','nids_v2_label_encoder.pkl',
             'nids_v2_feature_cols.npy','nids_v2_results.json']:
    print(' -', name)

## 8. Notes on what's still imperfect

Being honest about remaining limitations — the MLSecOps Agent project will catch some of these on the next pass:

- **Mapping unknown test-only attack subtypes to `R2L`** is a heuristic. A cleaner approach is to treat them as a separate `Unknown` class and report open-set detection metrics.
- **Model artifacts are saved with `joblib`/`.h5` and no integrity check.** `joblib.load` of an untrusted file is RCE-equivalent. Should be sealed with a SHA-256 manifest at minimum.
- **No adversarial robustness check.** A real IDS gets attacked. Trivial perturbations (e.g. padding `src_bytes`) may flip predictions. The MLSecOps Agent runs FGSM / boundary attacks against `nids_v2_lstm.h5` and reports evasion success rate.
- **Single split, no cross-validation.** Stratified k-fold on the (train+val) pool would give tighter confidence intervals on the macro-F1 numbers.
- **No calibration analysis.** The decision engine routes by confidence buckets, but the underlying softmax probabilities are not calibrated. A reliability diagram + temperature scaling would make the LOW/MEDIUM/HIGH/CRITICAL thresholds meaningful instead of arbitrary.